# Chapter 2: Polynomials - Examples

Jupyter notebook accompanying the *Polynomials* chapter.
Demonstrates polynomial division, gcd, and Lagrange interpolation.

## 1. Polynomial representation

We represent a polynomial $f(x) = a_0 + a_1 x + \cdots + a_n x^n$ as a list of its coefficients `[a_0, ..., a_n]` in ascending order.

In [ ]:
def poly_str(f):
    """Convert coefficient list (ascending) to readable string."""
    terms = []
    for i, c in enumerate(f):
        if c == 0:
            continue
        if i == 0:
            terms.append(f"{c}")
        elif i == 1:
            terms.append(f"{c}x")
        else:
            terms.append(f"{c}x^{i}")
    return " + ".join(terms) if terms else "0"

# f(x) = 1 + 2x + 0x^2 + 3x^3  -> [1, 2, 0, 3]
f = [1, 2, 0, 3]
print(poly_str(f))
print(f"degree = {len(f)-1}")

## 2. Polynomial division algorithm

Divide $f$ by $g$ over the rationals, obtaining quotient $q$ and remainder $r$ with $\deg r < \deg g$.

In [ ]:
from fractions import Fraction

def trim(p):
    while p and p[-1] == 0:
        p.pop()
    return p

def poly_divmod(f, g):
    """Polynomial long division. Returns (quotient, remainder).
    Coefficients as lists, ascending order."""
    f = list(f); g = list(g)
    trim(f); trim(g)
    if not g:
        raise ValueError("division by zero polynomial")
    q = [Fraction(0)] * max(len(f)-len(g)+1, 0)
    while f and len(f) >= len(g):
        # leading coefficient of quotient term
        lead = Fraction(f[-1], g[-1])
        q[len(f)-len(g)] = lead
        # subtract lead * g * x^{deg f - deg g}
        shift = len(f)-len(g)
        for i, c in enumerate(g):
            f[i+shift] -= lead * c
        trim(f)
    return (trim(q), f or [0])

f = [1, 0, 2, 1]   # 1 + 0x + 2x^2 + x^3
g = [1, 1]         # 1 + x
q, r = poly_divmod(f, g)
print(f"f = {poly_str(f)}")
print(f"g = {poly_str(g)}")
print(f"q = {poly_str(q)}")
print(f"r = {poly_str(r)}")
print("check: f = q*g + r :", poly_str(f), "=", poly_str(q), "*", poly_str(g), "+", poly_str(r))

## 3. Remainder theorem & roots

The remainder when dividing $f$ by $(x-\alpha)$ is exactly $f(\alpha)$. And $f$ has at most $\deg f$ roots.

In [ ]:
def eval_poly(f, x):
    """Horner's method."""
    result = 0
    for c in reversed(f):
        result = result * x + c
    return result

f = [1, 0, 2, 1]  # 1 + 2x^2 + x^3
alpha = 2
g = [-alpha, 1]  # x - 2
q, r = poly_divmod(f, g)
print(f"f(2) = {eval_poly(f, 2)}")
print(f"remainder of f/(x-2) = {r}")
print("They agree as expected!")

## 4. Polynomial gcd (Euclidean algorithm)

Compute $\gcd(f,g)$ via repeated division.

In [ ]:
def poly_gcd(f, g):
    while any(g):
        _, r = poly_divmod(f, g)
        f, g = g, r
    # make monic
    lc = f[-1]
    return [c / lc for c in f]

f = [0, 1, 1]    # x + x^2
g = [1, 1]       # 1 + x
d = poly_gcd(f, g)
print(f"gcd({poly_str(f)}, {poly_str(g)}) = {poly_str(d)}")

## 5. Lagrange interpolation

Given points $(x_i, y_i)$, find the unique polynomial of degree $\le n$ passing through all of them.

In [ ]:
def lagrange_basis(points, i, x):
    """Evaluate L_i(x) = prod_{j != i} (x - x_j)/(x_i - x_j)."""
    result = 1.0
    for j, (xj, _) in enumerate(points):
        if i != j:
            result *= (x - xj) / (points[i][0] - xj)
    return result

def lagrange_interp(points, x):
    return sum(yi * lagrange_basis(points, i, x) for i, (_, yi) in enumerate(points))

points = [(0, 1), (1, 3), (2, 7)]
for (xi, yi) in points:
    print(f"f({xi}) = {lagrange_interp(points, xi)} (expected {yi})")

# The interpolating polynomial is x^2 + x + 1
print("f(3) = ", lagrange_interp(points, 3), "(poly says 3^2+3+1 = 13)")

## 6. Lagrange over Z_p (Shamir secret sharing)

Splitting a secret into shares using a degree $t-1$ polynomial over a finite field.

In [ ]:
def modinv(a, p):
    return pow(a, -1, p)

def lagrange_share(points, x, p):
    result = 0
    for i, (xi, yi) in enumerate(points):
        num, den = 1, 1
        for j, (xj, _) in enumerate(points):
            if i != j:
                num = num * (x - xj) % p
                den = den * (xi - xj) % p
        Li = num * modinv(den, p) % p
        result = (result + yi * Li) % p
    return result

# Secret share with k=3 threshold, p=17
p = 17
secret = 11
# random polynomial f(x) = 11 + a1*x + a2*x^2
a1, a2 = 3, 5
def f(x): return (secret + a1*x + a2*x*x) % p
shares = [(i, f(i)) for i in range(1, 6)]  # give out 5 shares

# any 3 shares recover the secret f(0)
three = shares[:3]
recovered = lagrange_share(three, 0, p)
print(f"Secret: {secret}, recovered from 3 shares: {recovered}")

# 2 shares give nothing (all constants possible)
two = shares[:2]
print(f"Using only 2 shares, f(0) is not determined. Example value with same 2 shares: {lagrange_share(two, 0, p)}")